### RAG Pipeline


In [1]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


C:\Users\user\AppData\Local\Temp\ipykernel_9640\2607999595.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader
c:\Users\user\Desktop\LEGAL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_path = Path("../data/acts/civil_code.pdf")

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()

In [3]:
print(len(documents))

409


In [4]:
for doc in documents:
    doc.metadata.pop("producer",None)
    doc.metadata.pop("creator",None)
    doc.metadata['title'] = "The National Civil (Code) Act, 2017 (2074)"
    doc.metadata['source_file'] = "Civil Code"
    doc.metadata['file_type'] = 'pdf'


In [5]:
documents = documents[3:]


In [6]:
documents[0].page_content

'4 \n \nThe National Civil (Code) Act, 2017 (2074) \nDate of Authentication:  \n16 October 2017  \nAct Number 34 of the year 2017 \nAn Act Made To Amend And Consolidate Civil Laws \nPreamble: Whereas, it is expedient to make  timely the civil provisions \ncontained in the Country Code a nd other laws, by also amending and \nconsolidating such provisions, in or der to maintain mo rality, decency, \netiquette and convenience as well as economic interest of the public by \nmaintaining law and order in the country and maintain harmonious \nrelationship between various castes, races and communities, by making just \nprovisions in the economic, social and cultural fields; \nNow, therefore, the Legislature-Parliament under clause (1) of \nArticle 296 of the Constitution of Nepal has enacted this Act. \nPart-1 \nPreliminary \nChapter-1 \nGeneral Provisions \n1. Short title and commencement:  (1) This Act may be cited as the \n"National Civil (Code), 2017". \n(2) It shall come into force on 17 

In [7]:
import re
import json
from langchain_core.documents import Document

def parse_civil_code(civil_code_raw_text):
    # Split the raw document into lines
    civil_code_lines = civil_code_raw_text.split('\n')
    
    civil_code_chunks = []
    current_civil_part = "Unknown Part"
    current_civil_chapter = "Unknown Chapter"
    current_civil_section_title = ""
    current_civil_section_content = []
    
    # Regex patterns isolated for Civil Code formatting rules
    civil_part_pattern = re.compile(r'^Part\s*-\s*\d+', re.IGNORECASE)
    civil_chapter_pattern = re.compile(r'^Chapter\s*-\s*\d+', re.IGNORECASE)
    civil_section_pattern = re.compile(r'^(\d+)\.\s+(.*)') 
    
    # Page footer regex: Matches lines that contain ONLY digits (e.g., "69", " 70 ")
    civil_page_footer_pattern = re.compile(r'^\d+$')
    
    def save_civil_chunk():
        if current_civil_section_content:
            civil_chunk_text = " ".join(current_civil_section_content).strip()
            # Clean up double spaces caused by joins and line breaks
            civil_chunk_text = re.sub(r'\s+', ' ', civil_chunk_text)

            civil_code_chunks.append(
                Document(
                    page_content=civil_chunk_text,
                    metadata={
                        "title": "The National Civil (Code) Act, 2017 (2074)",
                        "doc_type": "Civil Code",
                        "part": current_civil_part,
                        "chapter": current_civil_chapter,
                        "section_title": current_civil_section_title,
                    }
                )
            )
    
    for i, line in enumerate(civil_code_lines):
        line_clean = line.strip()
        
        # 1. Skip empty lines
        if not line_clean:
            continue
            
        # 2. PRE-PROCESSING FILTER: Strip bare page-footer numbers (e.g., "69")
        # This prevents mid-clause text from getting broken by page breaks
        if civil_page_footer_pattern.match(line_clean):
            continue
            
        # 3. Detect and update the current Part
        if civil_part_pattern.match(line_clean):
            save_civil_chunk()
            current_civil_part = line_clean
            if i + 1 < len(civil_code_lines) and not re.match(r'^(Chapter|Part)', civil_code_lines[i+1].strip(), re.IGNORECASE):
                current_civil_part += " - " + civil_code_lines[i+1].strip()
            continue
            
        # 4. Detect and update the current Chapter
        if civil_chapter_pattern.match(line_clean):
            save_civil_chunk()
            current_civil_chapter = line_clean
            if i + 1 < len(civil_code_lines) and not re.match(r'^(\d+\.|Chapter|Part)', civil_code_lines[i+1].strip(), re.IGNORECASE):
                current_civil_chapter += " - " + civil_code_lines[i+1].strip()
            continue
            
        # 5. Detect a new Section
        section_match = civil_section_pattern.match(line_clean)
        if section_match:
            save_civil_chunk()
            current_civil_section_title = section_match.group(0)
            current_civil_section_content = [current_civil_section_title]
            continue
            
        # 6. Append text to the current section (if we are inside one)
        if current_civil_section_title:
            current_civil_section_content.append(line_clean)
            
    # Save the final chunk in the document
    save_civil_chunk() 
    return civil_code_chunks

In [8]:
civil_code_text = "\n".join(doc.page_content for doc in documents)

In [9]:
civil_code_chunks = parse_civil_code(civil_code_text)

In [10]:
print(len(civil_code_chunks))

775


In [11]:
print(civil_code_chunks[1])

page_content='2. Definitions: Unless the subject or the context otherwise requires, in this Act,- (a) “court” means the Supr eme Court, High Court or District Court, and this te rm includes a court, other judicial body or authority au thorized by law to proceed and adjudicate any specific type of civil cases, (b) "law" means a law in force for the time being, (c) "person" means a natural pe rson and this term includes a legal person, (d) "law-suit" means a statemen t of claims, and this term includes any other kind of complaint, claim, counterclaim or eq uivalent petition, (e) “minor” means a child who has not attained eighteen years of age, (f) "Local Level" means the Rural Municipality (Gownpalika) or Municipality, (g) "property" means any mova ble or immovable property, (h) "heir" means a person who is in the order of priority for succession pursuant to Section 239.' metadata={'title': 'The National Civil (Code) Act, 2017 (2074)', 'doc_type': 'Civil Code', 'part': 'Part-1 - Prelimin

In [12]:
pdf_path = Path("../data/acts/penal_code.pdf")

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()
print(len(documents))

200


In [13]:
for doc in documents:
    doc.metadata.pop("producer",None)
    doc.metadata.pop("creator",None)
    doc.metadata.pop("author")
    doc.metadata['title'] = "The National Penal (Code) Act, 2017"
    doc.metadata['source_file'] = "Penal Code"
    doc.metadata['file_type'] = 'pdf'


In [14]:
documents[0]

Document(metadata={'creationdate': '2018-12-07T13:13:09+05:30', 'moddate': '2020-08-25T12:41:50+02:00', 'title': 'The National Penal (Code) Act, 2017', 'source': '..\\data\\acts\\penal_code.pdf', 'total_pages': 200, 'page': 0, 'page_label': '1', 'source_file': 'Penal Code', 'file_type': 'pdf'}, page_content='1 \nRevised \nThe National Penal (Code) Act, 2017  \n \nDate of Authentication:  \n16 October 2017  \nAct number 36 of the year 2017 \nAn Act Made To Amend And Consolidate Laws In Force Relating To \nCriminal Offences \nPreamble:  \nWhereas, it is expedient to provide for a tim ely code on criminal \noffences, by amending and consolidating the laws in force relating to \ncriminal offences, in order to uphold morality, decency, etiquette, \nconvenience, economic interest of the public, by maintaining law and order \nin the country, mainta in harmonious relationship and peace among various \nreligious and cultural communities, and prevent and control criminal \noffences; \nNow, there

In [15]:

def parse_penal_code(penal_code_raw_text):
    # Split the raw document into lines
    lines = penal_code_raw_text.split('\n')
    
    chunks = []
    
    # Initialize defaults for preamble/introductory text
    current_part = "Preamble"
    current_chapter = "Preliminary"
    current_section_number = "0"
    current_section_title = "Introduction"
    current_section_content = []
    
    # Regex patterns optimized for Penal Code formatting
    part_pattern = re.compile(r'^Part\s*-\s*\d+', re.IGNORECASE)
    chapter_pattern = re.compile(r'^Chapter\s*-\s*\d+', re.IGNORECASE)
    section_pattern = re.compile(r'^(\d+)\.\s+(.*)') 
    
    # Page extraction artifacts: Isolates lines that are just numbers OR trailing numbers
    page_isolated_pattern = re.compile(r'^\s*\d+\s*$')
    page_trailing_pattern = re.compile(r'\s+\d+\s*$')
    
    def save_chunk():
        if current_section_content:
            chunk_text = " ".join(current_section_content).strip()
            # Clean up extra spaces
            chunk_text = re.sub(r'\s+', ' ', chunk_text)

            if chunk_text:
                chunks.append(
                    Document(
                        page_content=chunk_text,
                        metadata={
                            "title": "The National Penal (Code) Act, 2017",
                            "doc_type": "Penal Code",
                            "part": current_part,
                            "chapter": current_chapter,
                            "section_number": current_section_number,
                            "section_title": current_section_title,
                        }
                    )
                )
    
    for i, line in enumerate(lines):
        line_clean = line.strip()
        
        # 1. Skip completely empty lines or lines that are entirely just a page number
        if not line_clean or page_isolated_pattern.match(line_clean):
            continue
            
        # 2. Strip trailing page numbers appended to valid text lines
        line_clean = page_trailing_pattern.sub('', line_clean)
            
        # 3. Detect and update the current Part
        if part_pattern.match(line_clean):
            save_chunk()
            current_part = line_clean
            # Look ahead to capture the actual Part Title (e.g., "General Provisions")
            if i + 1 < len(lines):
                next_line = page_trailing_pattern.sub('', lines[i+1].strip())
                if not re.match(r'^(Chapter|Part|\d+\.)', next_line, re.IGNORECASE):
                    current_part += " - " + next_line
            
            # Reset section tracking for the new part
            current_section_title = "" 
            current_section_content = []
            continue
            
        # 4. Detect and update the current Chapter
        if chapter_pattern.match(line_clean):
            save_chunk()
            current_chapter = line_clean
            # Look ahead to capture the actual Chapter Title
            if i + 1 < len(lines):
                next_line = page_trailing_pattern.sub('', lines[i+1].strip())
                if not re.match(r'^(Chapter|Part|\d+\.)', next_line, re.IGNORECASE):
                    current_chapter += " - " + next_line
                    
            # Reset section tracking for the new chapter
            current_section_title = "" 
            current_section_content = []
            continue
            
        # 5. Detect a new Section
        section_match = section_pattern.match(line_clean)
        if section_match:
            save_chunk()
            current_section_number = section_match.group(1)
            current_section_title = line_clean 
            current_section_content = [line_clean]
            continue
            
        # 6. Prevent Part/Chapter titles from being duplicated into the content body
        if current_part.endswith(line_clean) or current_chapter.endswith(line_clean):
            continue
            
        # 7. Append text to the current section
        current_section_content.append(line_clean)
            
    # Save the final section when the loop finishes
    save_chunk() 
    return chunks



In [16]:
penal_code_text = "\n".join(doc.page_content for doc in documents)

In [17]:
penal_code_chunks = parse_penal_code(penal_code_text)

In [18]:
print(len(penal_code_chunks))

311


In [19]:
print(penal_code_chunks[1])

page_content='1. Short title and commencement: (1) This Act may be cited as the "National Penal (Code) Act, 2017". (2) It shall commence on 17 August 2018 (first day of the month of Bhadra of the year 2075).' metadata={'title': 'The National Penal (Code) Act, 2017', 'doc_type': 'Penal Code', 'part': 'Part -1 - General Provisions', 'chapter': 'Chapter-1 - Preliminary', 'section_number': '1', 'section_title': '1. Short title and commencement:  (1) This Act may be cited as the'}


In [20]:
pdf_path = Path("../data/acts/constitution.pdf")

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()
print(len(documents))

240


In [21]:
for doc in documents:
    doc.metadata.pop("producer",None)
    doc.metadata.pop("creator",None)
    doc.metadata['title'] = "THE CONSTITUTION OF NEPAL"
    doc.metadata['source_file'] = "Constitution"
    doc.metadata['file_type'] = 'pdf'

In [22]:
documents[0]

Document(metadata={'creationdate': '2015-11-01T10:54:39+05:45', 'source': '..\\data\\acts\\constitution.pdf', 'total_pages': 240, 'page': 0, 'page_label': '1', 'title': 'THE CONSTITUTION OF NEPAL', 'source_file': 'Constitution', 'file_type': 'pdf'}, page_content='1 \n \n \n \n \n \nTHE CONSTITUTION OF NEPAL')

In [23]:
documents = documents[5:]

In [24]:
documents[0].page_content

"6 \n \nThe Constitution of Nepal \nDate of Publication in Nepal Gazette \n20 September 2015 (2072.6.3) \nPreamble: \nWe, the Sovereign People of Nepal,  \nInternalizing the people's sovereign right and right to autonomy and self -rule, while \nmaintaining freedom, sovereignty, territorial integrity, national unity, independence \nand dignity of Nepal, \nRecalling the glorious history of historic people's movements, armed conflict, \ndedication and sacrifice undertaken by the Nepalese people at times for the interest of \nthe nation, democracy and progressive changes , and respecting for the martyrs and \ndisappeared and victim citizens, \nEnding all forms of discrimination and oppression created by the feudalis tic, \nautocratic, centralized, unitary system of governance, \nProtecting and  promoting social and cultural solidarity, tolerance and  harmony, and \nunity in diversity  by recognizing the multi -ethnic, multi -lingual, multi -religious, \nmulti-cultural and diverse r egional

In [25]:

def parse_constitution(constitution_raw_text):
    # 1. PRE-PROCESSING: Split into lines and strip bare page-footer numbers before parsing
    raw_lines = constitution_raw_text.split('\n')
    page_footer_pattern = re.compile(r'^\s*\d+\s*$')
    
    # Keep only lines that are not standalone page numbers
    constitution_lines = [line for line in raw_lines if not page_footer_pattern.match(line)]
    
    constitution_chunks = []
    current_constitution_part = "Preamble" # Defaults to Preamble for the document opening
    current_constitution_chapter = ""      # Kept empty unless explicitly found inside a Part
    current_constitution_article_title = ""
    current_constitution_article_content = []
    
    # Regex patterns adapted for Constitution formatting rules
    constitution_part_pattern = re.compile(r'^Part\s*-?\s*\d+', re.IGNORECASE)
    constitution_chapter_pattern = re.compile(r'^Chapter\s*-?\s*\d+', re.IGNORECASE)
    
    # Matches "Article 1. Title" OR just "1. Title"
    constitution_article_pattern = re.compile(r'^(?:Article\s+)?(\d+)\.\s+(.*)', re.IGNORECASE)
    
    # Matches "Schedule - 1" or "Schedule 1"
    constitution_schedule_pattern = re.compile(r'^Schedule\s*-?\s*\d+', re.IGNORECASE)
    
    def save_constitution_chunk():
        if current_constitution_article_content:
            constitution_chunk_text = " ".join(current_constitution_article_content).strip()
            # Clean up double spaces caused by joins and line breaks
            constitution_chunk_text = re.sub(r'\s+', ' ', constitution_chunk_text)

            constitution_chunks.append(
                Document(
                    page_content=constitution_chunk_text,
                    metadata={
                        "title": "THE CONSTITUTION OF NEPAL",
                        "doc_type": "Constitution",
                        "part": current_constitution_part,
                        "chapter": current_constitution_chapter,
                        "article_title": current_constitution_article_title,
                    }
                )
            )
    
    # 2. PARSING: Iterate through the cleaned lines
    for i, line in enumerate(constitution_lines):
        line_clean = line.strip()
        
        # Skip empty lines
        if not line_clean:
            continue
            
        # Detect and update Schedules (Treat them as unique Parts at the end)
        if constitution_schedule_pattern.match(line_clean):
            save_constitution_chunk()
            current_constitution_part = line_clean
            current_constitution_chapter = ""
            current_constitution_article_title = line_clean
            current_constitution_article_content = [line_clean]
            
            # Look ahead for the Schedule's title
            if i + 1 < len(constitution_lines) and not re.match(r'^(Schedule|Part|Article)', constitution_lines[i+1].strip(), re.IGNORECASE):
                schedule_title = constitution_lines[i+1].strip()
                current_constitution_part += " - " + schedule_title
                current_constitution_article_content.append(schedule_title)
            continue
            
        # Detect and update the current Part
        if constitution_part_pattern.match(line_clean):
            save_constitution_chunk()
            current_constitution_part = line_clean
            current_constitution_chapter = "" # Reset chapter when a new part begins
            
            # Look ahead for the Part's title
            if i + 1 < len(constitution_lines) and not re.match(r'^(Chapter|Part|Article|\d+\.)', constitution_lines[i+1].strip(), re.IGNORECASE):
                current_constitution_part += " - " + constitution_lines[i+1].strip()
            continue
            
        # Detect and update the current Chapter (If applicable)
        if constitution_chapter_pattern.match(line_clean):
            # We don't save chunk here because chapters wrap articles, we just update the metadata
            current_constitution_chapter = line_clean
            if i + 1 < len(constitution_lines) and not re.match(r'^(\d+\.|Article|Chapter|Part)', constitution_lines[i+1].strip(), re.IGNORECASE):
                current_constitution_chapter += " - " + constitution_lines[i+1].strip()
            continue
            
        # Detect a new Article
        article_match = constitution_article_pattern.match(line_clean)
        if article_match:
            save_constitution_chunk()
            current_constitution_article_title = line_clean
            current_constitution_article_content = [current_constitution_article_title]
            continue
            
        # Initial Preamble edge-case handling (Before any article is explicitly declared)
        if not current_constitution_article_title and current_constitution_part == "Preamble":
            if line_clean.lower() == "preamble":
                current_constitution_article_title = "Preamble"
                current_constitution_article_content = [line_clean]
            else:
                if not current_constitution_article_content:
                    current_constitution_article_title = "Preamble"
                current_constitution_article_content.append(line_clean)
            continue
            
        # Append text to the current article (if we are inside one)
        if current_constitution_article_title:
            current_constitution_article_content.append(line_clean)
            
    # Save the final chunk in the document
    save_constitution_chunk() 
    
    return constitution_chunks

In [26]:
constitution_text = "\n".join(doc.page_content for doc in documents)

In [27]:
constitution_chunks = parse_constitution(constitution_text)

In [28]:
print(len(constitution_chunks))

548


In [29]:
print(constitution_chunks[0])

page_content='The Constitution of Nepal Date of Publication in Nepal Gazette 20 September 2015 (2072.6.3) Preamble: We, the Sovereign People of Nepal, Internalizing the people's sovereign right and right to autonomy and self -rule, while maintaining freedom, sovereignty, territorial integrity, national unity, independence and dignity of Nepal, Recalling the glorious history of historic people's movements, armed conflict, dedication and sacrifice undertaken by the Nepalese people at times for the interest of the nation, democracy and progressive changes , and respecting for the martyrs and disappeared and victim citizens, Ending all forms of discrimination and oppression created by the feudalis tic, autocratic, centralized, unitary system of governance, Protecting and promoting social and cultural solidarity, tolerance and harmony, and unity in diversity by recognizing the multi -ethnic, multi -lingual, multi -religious, multi-cultural and diverse r egional characteristics, resolving to

In [30]:
# Combine all chunks into a single flat list
all_legal_chunks = civil_code_chunks + penal_code_chunks + constitution_chunks


print(f"Total chunks unified: {len(all_legal_chunks)}")
print(f"Sample chunk: {all_legal_chunks[900]}")

Total chunks unified: 1634
Sample chunk: page_content='124. Prohibition of committing public nuisance: (1) Except as otherwise provided in a law, no person shall do any act or omit to do any act legally required to be done, which causes any kind of harm, injury, danger or annoyance to the public or to the people who dwell in the vicinity. (2) A person who commits the offence referred to in sub- section (1) shall be liable to a sentence of fine not exceeding twenty - five thousand rupees.' metadata={'title': 'The National Penal (Code) Act, 2017', 'doc_type': 'Penal Code', 'part': 'Part-2 - Criminal Offences', 'chapter': 'Chapter-5 - Offences against Public Interest, Health, Safety,', 'section_number': '124', 'section_title': '124. Prohibition of committing public nuisance:  (1) Except as'}


### Embedding and Vector Store DB

In [31]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
class EmbeddingManager:
    
    """
    handles document embedding gerneration using SentenceTransformer
    """


    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise

    def generate_embeddings(self, texts: List[Document]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args: 
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        # Extract text from each Document
        page_contents = [doc.page_content for doc in texts]

        print(f"Generating embeddings for {len(page_contents)} documents...")
        embeddings = self.model.encode(
            page_contents,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")
        
        return embeddings
    
    def generate_query_embedding(self, query: str) -> np.ndarray:
        """
        Generate an embedding for a single query string.
        """
        if self.model is None:
            raise ValueError("Model not loaded")

        return self.model.encode(query)
    
### intialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


    


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1422.75it/s]


Model loaded successfully. Embedding dimension: 384


In [33]:
### Vector Store

class VectorStore:
    """ Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name:str="pdf_documents", persist_directory:str = "../data/vector_store"):
        """"
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        
        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents:List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store.....")

        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate (zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = {
                key: ("" if value is None else value)
                for key, value in doc.metadata.items()
            }

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            #Document content
            documents_text.append(doc.page_content)

            #Embedding 
            embeddings_list.append(embedding.tolist())
        
        
        #Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text

            )
            print(f"Succesfullt added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
try:
    vectorstore.client.delete_collection(vectorstore.collection_name)
    print("Old collection deleted.")
except Exception:
    print("Collection did not exist.")
vectorstore = VectorStore()
vectorstore

        
        
        

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1634
Old collection deleted.
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [34]:
all_legal_chunks

[Document(metadata={'title': 'The National Civil (Code) Act, 2017 (2074)', 'doc_type': 'Civil Code', 'part': 'Part-1 - Preliminary', 'chapter': 'Chapter-1 - General Provisions', 'section_title': '1. Short title and commencement:  (1) This Act may be cited as the'}, page_content='1. Short title and commencement: (1) This Act may be cited as the "National Civil (Code), 2017". (2) It shall come into force on 17 August 2018 (first day of the month of Bhadra of the year 2075).'),
 Document(metadata={'title': 'The National Civil (Code) Act, 2017 (2074)', 'doc_type': 'Civil Code', 'part': 'Part-1 - Preliminary', 'chapter': 'Chapter-1 - General Provisions', 'section_title': '2. Definitions:  Unless the subject or the context otherwise requires, in'}, page_content='2. Definitions: Unless the subject or the context otherwise requires, in this Act,- (a) “court” means the Supr eme Court, High Court or District Court, and this te rm includes a court, other judicial body or authority au thorized by 

In [35]:
print(len(all_legal_chunks))

1634


In [36]:
print(type(all_legal_chunks))
print(type(all_legal_chunks[0]))
print(all_legal_chunks[0])

<class 'list'>
<class 'langchain_core.documents.base.Document'>
page_content='1. Short title and commencement: (1) This Act may be cited as the "National Civil (Code), 2017". (2) It shall come into force on 17 August 2018 (first day of the month of Bhadra of the year 2075).' metadata={'title': 'The National Civil (Code) Act, 2017 (2074)', 'doc_type': 'Civil Code', 'part': 'Part-1 - Preliminary', 'chapter': 'Chapter-1 - General Provisions', 'section_title': '1. Short title and commencement:  (1) This Act may be cited as the'}


In [37]:
print(len(all_legal_chunks))


1634


In [38]:
### Generate embeddings
embeddings = embedding_manager.generate_embeddings(all_legal_chunks)

# Store in the vector database
vectorstore.add_documents(all_legal_chunks, embeddings)


Generating embeddings for 1634 documents...


Batches:   0%|          | 0/52 [00:00<?, ?it/s]

Batches: 100%|██████████| 52/52 [01:09<00:00,  1.34s/it]


Generated embeddings with shape: (1634, 384)
Adding 1634 documents to vector store.....
Succesfullt added 1634 documents to vector store
Total documents in collection: 1634


### Retrieval Pipeline From VectorStore

In [39]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager:EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings 
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float =0.0) -> List[Dict[str,Any]]:
        """Retrieve relevant documents for a query
        
        Returns: 
            List of dictionaries containing retrieved documents and metadata 
        """
        
        print(f"Retreiving documents for query: '{query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_query_embedding(query)
        
        #Search in vector store
        try:
            results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "document": Document(
                                page_content=document,
                                metadata=metadata
                            ),
                            "id": doc_id,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
        return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)





In [40]:
rag_retriever

In [41]:
rag_retriever.retrieve("Punishment for robbery")

Retreiving documents for query: 'Punishment for robbery
Top K: 5, Score threshold: 0.0
Retrieved 5 documents (after filtering)


[{'document': Document(metadata={'title': 'The National Penal (Code) Act, 2017', 'part': 'Part-2 - Criminal Offences', 'section_title': '244. Prohibition of robbery: (1) No person shall  commit, or cause to be', 'section_number': '244', 'doc_index': 1020, 'chapter': 'Chapter-20 - Offences Relating to Theft and Robbery', 'content_length': 1419, 'doc_type': 'Penal Code'}, page_content='244. Prohibition of robbery: (1) No person shall commit, or cause to be committed, robbery. (2) A person shall be deemed to commit robbery if he or she: (a) causes or attempts to cause to any person death or hurt or restraint or obstruction or fear or intimidation of instant death or of instant hurt, in order to the committing of theft or in committing the theft or in carrying away property obtained by the theft or to the escaping of arrest after the commission of the theft or commits theft carrying a deadly weapon, (b) obtains any money or economic ben efit through criminal extortion by putting any person

In [48]:
rag_retriever.retrieve("i am the president of ruling party of lower house of nepal government, i do have the 2/3rd majority in the lower house but i dont have any seats in the upper house. is it able to pass the laws acts without having any seats in the upper house, if not then how the operations will be done if i have the majority government in the lower house and i ahve the pm and minsiters and need to update the laws acts ordinances and all.")

Retreiving documents for query: 'i am the president of ruling party of lower house of nepal government, i do have the 2/3rd majority in the lower house but i dont have any seats in the upper house. is it able to pass the laws acts without having any seats in the upper house, if not then how the operations will be done if i have the majority government in the lower house and i ahve the pm and minsiters and need to update the laws acts ordinances and all.
Top K: 5, Score threshold: 0.0
Retrieved 5 documents (after filtering)


[{'document': Document(metadata={'content_length': 3192, 'doc_type': 'Constitution', 'part': 'Part-14 - State Legislature', 'chapter': '', 'article_title': '176. Composition of State Assembly : (1) Each State Assembly  shall consist of a', 'doc_index': 1279, 'title': 'THE CONSTITUTION OF NEPAL'}, page_content='176. Composition of State Assembly : (1) Each State Assembly shall consist of a number of members, as follows: (a) Members in a number that is twice a s many as the number of members elected to the House of Representatives from the concerned State, through the first past the post electoral system, (b) The number of members to be set under clause (a) shall be considered to be sixty percent, and the rest f orty percent members to be elected, through the proportional electoral system. (2) Election constituencies shall be set on the basis of geography and population as provided for in the Federal law, for the election to members under sub-clause (a) of clause (1). (3) Sixty percent m

### Integrate VectorDB context pipeline with LLM

In [42]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

In [43]:
def rag_llm(query, retriever, llm, top_k = 5, min_score = 0.2, return_context = False):
    """Rag pipeline with features:
        return answer, sources, confidence score
    """
    results = retriever.retrieve(query, top_k = top_k, score_threshold=min_score)
    if not results:
        return {'answer': "No relevant context found.", 'sources': [], 'confidence': 0.0, 'context': ''}

    ##Prepare context and sources
    context = "\n\n".join(
            f"""Document: {doc["document"].metadata.get("title", "Unknown")}
                Part: {doc["document"].metadata.get("part", "")}
                Chapter: {doc["document"].metadata.get("chapter", "")}
                Section: {doc["document"].metadata.get("section_title", "")}

                {doc["document"].page_content}
            """
            for doc in results
        )
    sources = []

    for doc in results:
        metadata = doc["document"].metadata

        sources.append({
            "source": metadata.get("title", metadata.get("source", "unknown")),
            "page": metadata.get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": doc["document"].page_content[:120] + "......"
        })

    #Generate answer
    
    prompt = f"""
    
        You are a legal assistant.

        Answer the user's question using ONLY the provided legal context.
        When answering, always mention:
        - the document name (Civil Code, Penal Code, Constitution)
        - the section/article number whenever available.

        If the context does not contain the answer, say:
        "I couldn't find the answer in the provided legal documents." \nContext: \n{context}\n\n Question: {query}\n\nAnswer:
            
        """
    confidence = max([doc['similarity_score'] for doc in results])
    response = llm.invoke(prompt)
    
    output = {
        'answer': response.content, 
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output





In [50]:
# Example of a question which vectorless rag might find difficult to address
result = rag_llm("""""", rag_retriever, llm, top_k=5, min_score=0.0, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retreiving documents for query: '
Top K: 5, Score threshold: 0.0
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [51]:
#Example where the min_score had to be reduced for retreival
result = rag_llm("""
                        lets say a party has it 2/3 seats of the parliament in the lower house and he wishes to pass th all the bills and acts but he doesnt have majority seats (10% of seats only) in the upper house.

                        how he can passes the bill
                        as he has formed the lower house government for 5 years.
                        if he is not able to pass the bill, then how the workings of government will be done, if his government (PM) wants new ammendments in laws acts bills.

                        what is the provision for this
                        if possible how the bills can be passed and be implemented.
                        give me the complete detailed analysis of this scenario.
                        it should include al the possible ways for it.
                        and also inlcude sources for it.
                 """, rag_retriever, llm, top_k=3, min_score=0.02, return_context=True)
print("Answer:", result['answer'])

Retreiving documents for query: '
                        lets say a party has it 2/3 seats of the parliament in the lower house and he wishes to pass th all the bills and acts but he doesnt have majority seats (10% of seats only) in the upper house.

                        how he can passes the bill
                        as he has formed the lower house government for 5 years.
                        if he is not able to pass the bill, then how the workings of government will be done, if his government (PM) wants new ammendments in laws acts bills.

                        what is the provision for this
                        if possible how the bills can be passed and be implemented.
                        give me the complete detailed analysis of this scenario.
                        it should include al the possible ways for it.
                        and also inlcude sources for it.
                 
Top K: 3, Score threshold: 0.02
Retrieved 3 documents (after filtering)
An

In [54]:
#Example where the min_score had to be reduced for retreival
result = rag_llm("""
                        What constitutional protections do citizens have when the police search their house?
                 """, rag_retriever, llm, top_k=3, min_score=0.00, return_context=True)
print("Answer:", result['answer'])

Retreiving documents for query: '
                        What constitutional protections do citizens have when the police search their house?
                 
Top K: 3, Score threshold: 0.0
Retrieved 1 documents (after filtering)
Answer: I couldn't find the answer in the provided legal documents. The provided context only refers to the National Penal Code Act, 2017, specifically Section 301, which discusses the prohibition of searching another person's body, vehicle, or personal belongings without consent. It does not mention searches of houses or constitutional protections.
